# FGSM Vision Engine - Colab Training
Just run the cells below! The notebook will automatically download the dataset from your AWS S3 bucket, mount your Google Drive, and save checkpoints directly to your Drive so you don't lose them if Colab disconnects.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/fgsm_runs

In [ ]:
%cd /content
# Paste a freshly-generated presigned URL here at runtime - never hardcode
# one in this notebook. Generate one with:
#   aws s3 presign s3://fgsm-vision-models-aibridix-official/fgsm_colab_package.zip --expires-in 3600 --profile aibridix_official
# A prior version of this cell had a long-lived presigned URL (and the AWS
# access key it embedded) committed directly to git, which is very likely
# what got the old AWS account suspended for credential exposure.
from getpass import getpass
package_url = getpass("Presigned S3 URL for fgsm_colab_package.zip: ")
!wget -O fgsm_colab_package.zip "{package_url}"
!unzip -o -q fgsm_colab_package.zip


In [ ]:
%cd /content/fgsm_colab_package
!pip install -r requirements.txt

In [ ]:
# Patch the script to fix the warmup_ratio bug
!sed -i '/warmup_ratio/d' src/train_temporal_classifier.py
!python src/train_temporal_classifier.py --config configs/temporal_classifier.yaml
!cp -r runs/temporal_move_classifier /content/drive/MyDrive/fgsm_runs/

In [ ]:
# We patch the script to use max_steps=500 instead of 50 epochs to drop the time from 23 hours to ~1.5 hours
!sed -i 's/num_train_epochs=50/max_steps=500/g' src/train_hitbox_segmentation.py
!python src/train_hitbox_segmentation.py --data data/ufd/segmentation_dataset --output /content/drive/MyDrive/fgsm_runs/hitbox_segmenter

In [ ]:
!echo 'Training complete! Checkpoints are saved in your Google Drive under MyDrive/fgsm_runs/ !'